**Search engine With tools agents**

In [1]:
#installing libraries

!pip install langchain-google-genai
!pip install langchain-community
!pip install yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 14.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.0
    Uninstalling langchain-core-1.6.0:
      Successfully uninstalled langchain-core-1.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [28]:
# import libraries
import warnings
warnings.filterwarnings("ignore")
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_community.tools import YahooFinanceNewsTool

from dotenv import load_dotenv
load_dotenv()
from google.colab import userdata
import os

In [3]:
from langchain_core.prompts import ChatPromptTemplate

In [8]:
!pip install langgraph

In [13]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

In [37]:
# Structured output
class FinanceNews(BaseModel):
  summary:str = Field(description="Summary of the latest financial news")
  key_points:list[str]
  sentiment:str
  ticker:str

# Model
model = ChatGoogleGenerativeAI(model='gemini-1.5-flash',api_key=userdata.get("GEMINI"),temperature=0)

#Yahoo finance tool

news_tool = YahooFinanceNewsTool(top_k=5,name="yahoo_finance_news_research")

system_prompt = ChatPromptTemplate.from_messages([
                                                              ("system",""" You are financial news reseacrh agent.
                                                               Use the Yahoo finance news tool for the current company news.
                                                               Summarize the important informatation concisely.
                                                               Do not invent financial information """),

                                                              ("user","{question}")
                                                  ])

#agent

agent = create_agent(
    tools=[news_tool],
    model=model,
    response_format=FinanceNews,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
#streaming
for chunk in agent.stream(
    {
        "messages":[
            {
                "role":"user",
                "content": "Find the latest news about Infosys. use the ticker INFY."
            }
        ]
    },
    config={"configurable": {"thread_id": "1"}},
    stream_mode="updates"
):
    print(chunk)

In [ ]:
#Clean result
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find the latest news about infosys. Use ticker INFY."
            }
        ]
    },
    config={"configurable": {"thread_id": "1"}}
)

print(result["structured_response"])

In [ ]:
#Batch processing
quieries = [
    {"messages":[{"role":"user","content":"Find the latest news about Infosys. use the ticker INFY."}]},
    {"messages":[{"role":"user","content":"Find the latest news about IBM. Use ticker IBM."}]},
    {"messages":[{"role":"user","content":"Find the latest news about Tesla. Use ticker TSLA."}]},
    {"messages":[{"role":"user","content":"Find the latest news about Microsoft. Use ticker MSFT."}]},
]

results = agent.batch(quieries)

for result in results:
  print(result["structured_response"])